In [3]:
from pathlib import Path
import json
import random
import math
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer


In [4]:
REPO_ROOT = Path.cwd()

DATA_DIR = REPO_ROOT
PROCESSED_DIR = DATA_DIR / "processed"
STATS_DIR = DATA_DIR / "stats"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Stats dir:", STATS_DIR)

Repo root: /home/klyuliana/code/DPO_ME_IF_YOU_CAN/data
Processed dir: /home/klyuliana/code/DPO_ME_IF_YOU_CAN/data/processed
Stats dir: /home/klyuliana/code/DPO_ME_IF_YOU_CAN/data/stats


### Config

In [5]:
SEED = 42

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_PROMPT = 256
MAX_TOTAL = 512
MIN_RESP = 8

MIN_LEN_RATIO = 0.5
MAX_LEN_RATIO = 2.0

# Stricter version, if needed:
# MIN_LEN_RATIO = 0.67
# MAX_LEN_RATIO = 1.5

N_TRAIN = 3000
N_VAL = 500
N_TEST = 500

random.seed(SEED)
np.random.seed(SEED)


### Utils

In [ ]:
tok = None

def tok_len(text: str) -> int:
    if text is None:
        return 0
    return len(tok(str(text), add_special_tokens=False)["input_ids"])

def clean_text(x: Any) -> str:
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\r\n", "\n").replace("\r", "\n")
    x = x.strip()
    return x

def write_jsonl(rows: List[Dict[str, Any]], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(rows):,} rows to {path}")

def save_json(obj: Dict[str, Any], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f"Wrote stats to {path}")

def load_dahoas_rm_static(split: str = "train") -> List[Dict[str, str]]:
    ds = load_dataset("Dahoas/rm-static", split=split)

    print(ds)
    print("Columns:", ds.column_names)
    print("Example:", ds[0])

    rows = []
    for i, ex in enumerate(ds):
        # Common columns for Dahoas/rm-static:
        # prompt, chosen, rejected
        prompt = clean_text(ex.get("prompt", ""))
        chosen = clean_text(ex.get("chosen", ""))
        rejected = clean_text(ex.get("rejected", ""))

        if prompt and chosen and rejected:
            rows.append({
                "source": "Dahoas/rm-static",
                "source_id": i,
                "prompt": prompt,
                "chosen": chosen,
                "rejected": rejected,
            })

    return rows

def load_orca_dpo_pairs(split: str = "train") -> List[Dict[str, str]]:
    ds = load_dataset("Intel/orca_dpo_pairs", split=split)

    print(ds)
    print("Columns:", ds.column_names)
    print("Example:", ds[0])

    rows = []
    for i, ex in enumerate(ds):
        prompt = (
            ex.get("prompt")
            or ex.get("question")
            or ex.get("instruction")
            or ex.get("input")
            or ""
        )
        chosen = ex.get("chosen") or ex.get("response_chosen") or ex.get("accepted") or ""
        rejected = ex.get("rejected") or ex.get("response_rejected") or ex.get("rejected_response") or ""

        prompt = clean_text(prompt)
        chosen = clean_text(chosen)
        rejected = clean_text(rejected)

        if prompt and chosen and rejected:
            rows.append({
                "source": "Intel/orca_dpo_pairs",
                "source_id": i,
                "prompt": prompt,
                "chosen": chosen,
                "rejected": rejected,
            })

    return rows

In [13]:
def add_length_stats(row: Dict[str, Any]) -> Dict[str, Any]:
    p = clean_text(row["prompt"])
    c = clean_text(row["chosen"])
    r = clean_text(row["rejected"])

    pl = tok_len(p)
    cl = tok_len(c)
    rl = tok_len(r)

    out = dict(row)
    out["prompt"] = p
    out["chosen"] = c
    out["rejected"] = r

    out["prompt_len"] = pl
    out["chosen_len"] = cl
    out["rejected_len"] = rl
    out["chosen_total_len"] = pl + cl
    out["rejected_total_len"] = pl + rl
    out["chosen_longer"] = cl > rl
    out["len_ratio_chosen_rejected"] = cl / max(rl, 1)
    out["would_truncate_chosen"] = (pl > MAX_PROMPT) or (pl + cl > MAX_TOTAL)
    out["would_truncate_rejected"] = (pl > MAX_PROMPT) or (pl + rl > MAX_TOTAL)
    out["would_truncate_any"] = out["would_truncate_chosen"] or out["would_truncate_rejected"]

    return out

def percentile(x, q):
    if len(x) == 0:
        return None
    return float(np.percentile(np.array(x), q))


def compute_dataset_stats(rows: List[Dict[str, Any]], name: str = "dataset") -> Dict[str, Any]:
    if len(rows) == 0:
        return {"name": name, "n": 0}

    df = pd.DataFrame(rows)

    stats = {
        "name": name,
        "n": int(len(df)),

        "prompt_len": {
            "mean": float(df["prompt_len"].mean()),
            "median": float(df["prompt_len"].median()),
            "p90": percentile(df["prompt_len"], 90),
            "p95": percentile(df["prompt_len"], 95),
            "max": int(df["prompt_len"].max()),
        },

        "chosen_len": {
            "mean": float(df["chosen_len"].mean()),
            "median": float(df["chosen_len"].median()),
            "p90": percentile(df["chosen_len"], 90),
            "p95": percentile(df["chosen_len"], 95),
            "max": int(df["chosen_len"].max()),
        },

        "rejected_len": {
            "mean": float(df["rejected_len"].mean()),
            "median": float(df["rejected_len"].median()),
            "p90": percentile(df["rejected_len"], 90),
            "p95": percentile(df["rejected_len"], 95),
            "max": int(df["rejected_len"].max()),
        },

        "chosen_total_len": {
            "mean": float(df["chosen_total_len"].mean()),
            "median": float(df["chosen_total_len"].median()),
            "p90": percentile(df["chosen_total_len"], 90),
            "p95": percentile(df["chosen_total_len"], 95),
            "max": int(df["chosen_total_len"].max()),
        },

        "rejected_total_len": {
            "mean": float(df["rejected_total_len"].mean()),
            "median": float(df["rejected_total_len"].median()),
            "p90": percentile(df["rejected_total_len"], 90),
            "p95": percentile(df["rejected_total_len"], 95),
            "max": int(df["rejected_total_len"].max()),
        },

        "length_bias": {
            "chosen_longer_frac": float(df["chosen_longer"].mean()),
            "chosen_longer_percent": float(100 * df["chosen_longer"].mean()),
            "mean_chosen_minus_rejected": float((df["chosen_len"] - df["rejected_len"]).mean()),
            "median_chosen_minus_rejected": float((df["chosen_len"] - df["rejected_len"]).median()),
            "mean_len_ratio_chosen_rejected": float(df["len_ratio_chosen_rejected"].mean()),
            "median_len_ratio_chosen_rejected": float(df["len_ratio_chosen_rejected"].median()),
        },

        "truncation": {
            "chosen_truncation_frac": float(df["would_truncate_chosen"].mean()),
            "rejected_truncation_frac": float(df["would_truncate_rejected"].mean()),
            "any_truncation_frac": float(df["would_truncate_any"].mean()),
            "any_truncation_percent": float(100 * df["would_truncate_any"].mean()),
        },

        "quality": {
            "empty_prompt_frac": float((df["prompt"].str.len() == 0).mean()),
            "empty_chosen_frac": float((df["chosen"].str.len() == 0).mean()),
            "empty_rejected_frac": float((df["rejected"].str.len() == 0).mean()),
            "duplicate_prompt_frac": float(1.0 - df["prompt"].nunique() / max(len(df), 1)),
        }
    }

    return stats

def stats_to_table(stats: Dict[str, Any]) -> pd.DataFrame:
    rows = []

    for field in ["prompt_len", "chosen_len", "rejected_len", "chosen_total_len", "rejected_total_len"]:
        s = stats[field]
        rows.append({
            "field": field,
            "mean": s["mean"],
            "median": s["median"],
            "p90": s["p90"],
            "p95": s["p95"],
            "max": s["max"],
        })

    return pd.DataFrame(rows)

In [21]:
def keep(row: Dict[str, Any]) -> bool:
    p = row["prompt"]
    c = row["chosen"]
    r = row["rejected"]

    # Non-empty strings.
    if not p or not c or not r:
        return False

    pl = row["prompt_len"]
    cl = row["chosen_len"]
    rl = row["rejected_len"]

    # Prompt length.
    if pl > MAX_PROMPT:
        return False

    # Total length.
    if pl + cl > MAX_TOTAL:
        return False
    if pl + rl > MAX_TOTAL:
        return False

    # Minimum response length.
    if cl < MIN_RESP or rl < MIN_RESP:
        return False

    # Length ratio.
    ratio = cl / max(rl, 1)
    if not (MIN_LEN_RATIO <= ratio <= MAX_LEN_RATIO):
        return False

    return True

In [23]:
def acceptance_report(stats: Dict[str, Any], n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST):
    n_needed = n_train + n_val + n_test

    checks = []

    def add_check(name, value, target, passed):
        checks.append({
            "check": name,
            "value": value,
            "target": target,
            "passed": bool(passed),
        })

    add_check(
        "Enough examples",
        stats["n"],
        f">= {n_needed}",
        stats["n"] >= n_needed,
    )

    add_check(
        "Truncation rate",
        stats["truncation"]["any_truncation_percent"],
        "< 20%",
        stats["truncation"]["any_truncation_percent"] < 20,
    )

    add_check(
        "Chosen longer %",
        stats["length_bias"]["chosen_longer_percent"],
        "45–70% ideal, warn if >75%",
        stats["length_bias"]["chosen_longer_percent"] <= 75,
    )

    add_check(
        "Median prompt len",
        stats["prompt_len"]["median"],
        "< 128",
        stats["prompt_len"]["median"] < 128,
    )

    add_check(
        "P90 prompt len",
        stats["prompt_len"]["p90"],
        "< 256",
        stats["prompt_len"]["p90"] < 256,
    )

    add_check(
        "Median chosen len",
        stats["chosen_len"]["median"],
        "30–150 ideal",
        30 <= stats["chosen_len"]["median"] <= 150,
    )

    add_check(
        "Median rejected len",
        stats["rejected_len"]["median"],
        "30–150 ideal",
        30 <= stats["rejected_len"]["median"] <= 150,
    )

    return pd.DataFrame(checks)


In [27]:
def show_example(row):
    print("=" * 100)
    print(f"source: {row.get('source')} | source_id: {row.get('source_id')}")
    print(f"prompt_len={row['prompt_len']}, chosen_len={row['chosen_len']}, rejected_len={row['rejected_len']}")
    print(f"ratio={row['len_ratio_chosen_rejected']:.2f}")
    print("-" * 100)
    print("PROMPT:")
    print(row["prompt"])
    print("-" * 100)
    print("CHOSEN:")
    print(row["chosen"])
    print("-" * 100)
    print("REJECTED:")
    print(row["rejected"])
    print("=" * 100)


## Loading shit
1) model
2) dataset

### Model / Tokenizer

In [6]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Tokenizer loaded:", MODEL_NAME)
print("EOS:", tok.eos_token, tok.eos_token_id)
print("PAD:", tok.pad_token, tok.pad_token_id)


Tokenizer loaded: Qwen/Qwen2.5-0.5B-Instruct
EOS: <|im_end|> 151645
PAD: <|endoftext|> 151643


### Dahoas/rm-static

In [16]:
rm_rows_raw = load_dahoas_rm_static("train")
print("Loaded rows:", len(rm_rows_raw))
rm_rows_raw[0]


Dataset({
    features: ['prompt', 'response', 'chosen', 'rejected'],
    num_rows: 76256
})
Columns: ['prompt', 'response', 'chosen', 'rejected']
Example: {'prompt': '\n\nHuman: Can you describe the steps to clean fingerprints and smudges from a laptop screen\n\nAssistant: Yes, certainly. To clean your screen, you first need to use a microfiber cloth or soft, damp cloth to gently wipe down the surface of the screen. Next, you’ll want to grab a soft, lint-free, microfiber cleaning cloth and gently rub it back and forth across the screen to remove fingerprints and smudges.\n\nHuman: Can I spray isopropyl alcohol onto the cloth and clean it that way?\n\nAssistant:', 'response': ' Yes, you can do that to help the cloth pick up even more dirt from the screen. Be sure to always use a clean, soft cloth, not a piece of scratchy, roughened, or textured material, and make sure it’s lint-free.', 'chosen': ' Yes, you can do that to help the cloth pick up even more dirt from the screen. Be sure to

{'source': 'Dahoas/rm-static',
 'source_id': 0,
 'prompt': 'Human: Can you describe the steps to clean fingerprints and smudges from a laptop screen\n\nAssistant: Yes, certainly. To clean your screen, you first need to use a microfiber cloth or soft, damp cloth to gently wipe down the surface of the screen. Next, you’ll want to grab a soft, lint-free, microfiber cleaning cloth and gently rub it back and forth across the screen to remove fingerprints and smudges.\n\nHuman: Can I spray isopropyl alcohol onto the cloth and clean it that way?\n\nAssistant:',
 'chosen': 'Yes, you can do that to help the cloth pick up even more dirt from the screen. Be sure to always use a clean, soft cloth, not a piece of scratchy, roughened, or textured material, and make sure it’s lint-free.',
 'rejected': 'Yes, you can spray it directly onto the cloth.'}

In [18]:
rows_stats = [add_length_stats(r) for r in tqdm(rm_rows_raw)]
df = pd.DataFrame(rows_stats)

df.head()

,source,source_id,prompt,chosen,rejected,prompt_len,chosen_len,rejected_len,chosen_total_len,rejected_total_len,chosen_longer,len_ratio_chosen_rejected,would_truncate_chosen,would_truncate_rejected,would_truncate_any
0,Dahoas/rm-static,0,Human: Can you describe the steps to clean fin...,"Yes, you can do that to help the cloth pick up...","Yes, you can spray it directly onto the cloth.",110,52,11,162,121,True,4.727273,False,False,False
1,Dahoas/rm-static,1,Human: What are some foods that are good for d...,What exactly are you asking? There’s a lot of ...,"Sure, we’ve got information on common mistakes...",90,48,55,138,145,False,0.872727,False,False,False
2,Dahoas/rm-static,2,Human: What animal would be the dominate life ...,Possibly. They would definitely be very strong...,Insects and bacteria don’t move around in the ...,50,51,44,101,94,True,1.159091,False,False,False
3,Dahoas/rm-static,3,Human: How often are the Olympics?\n\nAssistant:,It is estimated that the Olympics occur every ...,"For 2017, it was every four years.",10,18,13,28,23,True,1.384615,False,False,False
4,Dahoas/rm-static,4,Human: Can I use car wax on my linoleum floor ...,What sort of flooring is it? This might be a ...,I’m not sure. Can you tell me more about what ...,20,46,16,66,36,True,2.875000,False,False,False


In [19]:
raw_stats = compute_dataset_stats(rows_stats, name="Dahoas/rm-static")
stats_to_table(raw_stats)

,field,mean,median,p90,p95,max
0,prompt_len,133.736096,101.0,306.0,382.0,1242
1,chosen_len,69.164983,54.0,146.0,191.0,750
2,rejected_len,57.745637,41.0,129.0,174.0,701
3,chosen_total_len,202.901079,171.0,394.0,480.0,1268
4,rejected_total_len,191.481733,158.0,387.0,473.0,1274


In [24]:
rows_filtered = [r for r in rows_stats if keep(r)]

print(f"Raw:      {len(rows_stats):,}")
print(f"Filtered: {len(rows_filtered):,}")
print(f"Kept:     {len(rows_filtered) / max(len(rows_stats), 1):.2%}")

Raw:      76,202
Filtered: 32,202
Kept:     42.26%


In [25]:
filtered_stats = compute_dataset_stats(rows_filtered, name="filtered_dataset")
print(json.dumps(filtered_stats, indent=2, ensure_ascii=False))

{
  "name": "filtered_dataset",
  "n": 32202,
  "prompt_len": {
    "mean": 95.16210173281162,
    "median": 85.0,
    "p90": 207.0,
    "p95": 229.0,
    "max": 256
  },
  "chosen_len": {
    "mean": 64.90811129743494,
    "median": 54.0,
    "p90": 125.0,
    "p95": 158.0,
    "max": 416
  },
  "rejected_len": {
    "mean": 63.68169678901931,
    "median": 52.0,
    "p90": 126.0,
    "p95": 160.0,
    "max": 496
  },
  "chosen_total_len": {
    "mean": 160.07021303024658,
    "median": 148.0,
    "p90": 285.0,
    "p95": 319.0,
    "max": 497
  },
  "rejected_total_len": {
    "mean": 158.84379852183093,
    "median": 147.0,
    "p90": 284.0,
    "p95": 318.0,
    "max": 511
  },
  "length_bias": {
    "chosen_longer_frac": 0.5133532078752873,
    "chosen_longer_percent": 51.33532078752873,
    "mean_chosen_minus_rejected": 1.2264145084156264,
    "median_chosen_minus_rejected": 1.0,
    "mean_len_ratio_chosen_rejected": 1.1015848920705347,
    "median_len_ratio_chosen_rejected": 1.0

In [26]:
acceptance_df = acceptance_report(filtered_stats)
acceptance_df

,check,value,target,passed
0,Enough examples,32202.000000,>= 4000,True
1,Truncation rate,0.000000,< 20%,True
2,Chosen longer %,51.335321,"45–70% ideal, warn if >75%",True
3,Median prompt len,85.000000,< 128,True
4,P90 prompt len,207.000000,< 256,True
5,Median chosen len,54.000000,30–150 ideal,True
6,Median rejected len,52.000000,30–150 ideal,True


In [28]:
for row in random.sample(rows_filtered, k=min(3, len(rows_filtered))):
    show_example(row)

source: Dahoas/rm-static | source_id: 49688
prompt_len=18, chosen_len=19, rejected_len=23
ratio=0.83
----------------------------------------------------------------------------------------------------
PROMPT:
Human: Whenever I have a salad it hurts my stomach. Why is this?

Assistant:
----------------------------------------------------------------------------------------------------
CHOSEN:
Do you mind if I take a moment to review some details about you and your health history?
----------------------------------------------------------------------------------------------------
REJECTED:
I’m not sure. I don’t think there are any scientific explanations for the relationship between salad and stomach pain.
source: Dahoas/rm-static | source_id: 8562
prompt_len=78, chosen_len=18, rejected_len=17
ratio=1.06
----------------------------------------------------------------------------------------------------
PROMPT:
Human: What are some cool things in the forest I can take pictures of?

Assi

### Anthropic HH-RLHF

In [10]:
def split_hh_conversation(text: str):
    """
    Splits an Anthropic HH-RLHF conversation into:
    context_before_final_assistant, final_assistant_response

    This is heuristic but usually works.
    """
    text = clean_text(text)
    marker = "\n\nAssistant:"
    
    idx = text.rfind(marker)
    if idx == -1:
        # Try less strict marker.
        marker = "Assistant:"
        idx = text.rfind(marker)
    
    if idx == -1:
        return "", ""

    prompt = text[:idx].strip()
    response = text[idx + len(marker):].strip()
    return prompt, response


def load_anthropic_hh(split: str = "train", subset: Optional[str] = None) -> List[Dict[str, str]]:
    """
    subset can be:
    - None for full
    - "helpful-base"
    - "harmless-base"
    depending on availability.
    """
    if subset is None:
        ds = load_dataset("Anthropic/hh-rlhf", split=split)
        source_name = "Anthropic/hh-rlhf"
    else:
        ds = load_dataset("Anthropic/hh-rlhf", data_dir=subset, split=split)
        source_name = f"Anthropic/hh-rlhf/{subset}"

    print(ds)
    print("Columns:", ds.column_names)
    print("Example:", ds[0])

    rows = []
    for i, ex in enumerate(ds):
        chosen_full = clean_text(ex.get("chosen", ""))
        rejected_full = clean_text(ex.get("rejected", ""))

        prompt_c, chosen = split_hh_conversation(chosen_full)
        prompt_r, rejected = split_hh_conversation(rejected_full)

        # Prefer prompt extracted from chosen.
        prompt = prompt_c or prompt_r

        # Optional sanity: if prompts differ too much, still keep chosen prompt.
        if prompt and chosen and rejected:
            rows.append({
                "source": source_name,
                "source_id": i,
                "prompt": prompt,
                "chosen": chosen,
                "rejected": rejected,
            })

    return rows


In [ ]:
# Full train split
# rows_raw = load_anthropic_hh("train")

# Or try harmless/helpful if available in your environment:
# rows_raw = load_anthropic_hh("train", subset="helpful-base")
# rows_raw = load_anthropic_hh("train", subset="harmless-base")


### Intel/orca_dpo_pairs

In [ ]:
orca_rows_raw = load_orca_dpo_pairs("train")

Generating train split: 100%|██████████| 12859/12859 [00:00<00:00, 47302.42 examples/s]


Dataset({
    features: ['system', 'question', 'chosen', 'rejected'],
    num_rows: 12859
})
Columns: ['system', 'question', 'chosen', 'rejected']
Example: {'system': '', 'question': "You will be given a definition of a task first, then some input of the task.\nThis task is about using the specified sentence and converting the sentence to Resource Description Framework (RDF) triplets of the form (subject, predicate object). The RDF triplets generated must be such that the triplets accurately capture the structure and semantics of the input sentence. The input is a sentence and the output is a list of triplets of the form [subject, predicate, object] that capture the relationships present in the sentence. When a sentence has more than 1 RDF triplet possible, the output must contain all of them.\n\nAFC Ajax (amateurs)'s ground is Sportpark De Toekomst where Ajax Youth Academy also play.\nOutput:", 'chosen': '[\n  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],\n  ["Ajax Y

### Saving dataset

In [29]:
def strip_for_training(row: Dict[str, Any], new_id: int) -> Dict[str, Any]:
    return {
        "id": int(new_id),
        "prompt": clean_text(row["prompt"]),
        "chosen": clean_text(row["chosen"]),
        "rejected": clean_text(row["rejected"]),
        "source": row.get("source", None),
        "source_id": row.get("source_id", None),
    }


In [30]:
rows_shuffled = rows_filtered.copy()
random.Random(SEED).shuffle(rows_shuffled)

n_available = len(rows_shuffled)
n_needed = N_TRAIN + N_VAL + N_TEST

print("Available:", n_available)
print("Needed:", n_needed)

if n_available < n_needed:
    print("⚠️ Not enough examples for requested split sizes.")
    print("Using reduced split sizes.")

    # Keep val/test at 300 if possible, train gets the rest.
    reduced_val = min(300, max(0, n_available // 5))
    reduced_test = min(300, max(0, n_available // 5))
    reduced_train = n_available - reduced_val - reduced_test

    N_TRAIN_FINAL = reduced_train
    N_VAL_FINAL = reduced_val
    N_TEST_FINAL = reduced_test
else:
    N_TRAIN_FINAL = N_TRAIN
    N_VAL_FINAL = N_VAL
    N_TEST_FINAL = N_TEST

print("Final split sizes:")
print("Train:", N_TRAIN_FINAL)
print("Val:  ", N_VAL_FINAL)
print("Test: ", N_TEST_FINAL)


Available: 32202
Needed: 4000
Final split sizes:
Train: 3000
Val:   500
Test:  500


In [31]:
train_raw = rows_shuffled[:N_TRAIN_FINAL]
val_raw = rows_shuffled[N_TRAIN_FINAL:N_TRAIN_FINAL + N_VAL_FINAL]
test_raw = rows_shuffled[N_TRAIN_FINAL + N_VAL_FINAL:N_TRAIN_FINAL + N_VAL_FINAL + N_TEST_FINAL]

train_rows = [strip_for_training(r, i) for i, r in enumerate(train_raw)]
val_rows = [strip_for_training(r, i) for i, r in enumerate(val_raw)]
test_rows = [strip_for_training(r, i) for i, r in enumerate(test_raw)]

len(train_rows), len(val_rows), len(test_rows)


(3000, 500, 500)

In [32]:
train_stats_rows = [add_length_stats(r) for r in train_rows]
val_stats_rows = [add_length_stats(r) for r in val_rows]
test_stats_rows = [add_length_stats(r) for r in test_rows]

split_stats = {
    "config": {
        "model_name_for_tokenizer": MODEL_NAME,
        "max_prompt": MAX_PROMPT,
        "max_total": MAX_TOTAL,
        "min_resp": MIN_RESP,
        "min_len_ratio": MIN_LEN_RATIO,
        "max_len_ratio": MAX_LEN_RATIO,
        "seed": SEED,
    },
    "raw": raw_stats,
    "filtered": filtered_stats,
    "splits": {
        "train": compute_dataset_stats(train_stats_rows, name="train"),
        "val": compute_dataset_stats(val_stats_rows, name="val"),
        "test": compute_dataset_stats(test_stats_rows, name="test"),
    }
}


In [33]:
write_jsonl(train_rows, PROCESSED_DIR / "train.jsonl")
write_jsonl(val_rows, PROCESSED_DIR / "val.jsonl")
write_jsonl(test_rows, PROCESSED_DIR / "test.jsonl")

save_json(split_stats, STATS_DIR / "dataset_stats.json")


Wrote 3,000 rows to /home/klyuliana/code/DPO_ME_IF_YOU_CAN/data/processed/train.jsonl
Wrote 500 rows to /home/klyuliana/code/DPO_ME_IF_YOU_CAN/data/processed/val.jsonl
Wrote 500 rows to /home/klyuliana/code/DPO_ME_IF_YOU_CAN/data/processed/test.jsonl
Wrote stats to /home/klyuliana/code/DPO_ME_IF_YOU_CAN/data/stats/dataset_stats.json


### Testing base models

In [34]:
import torch
from transformers import AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [35]:
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
)

base_model.eval()

 82%|████████▏ | 62712/76202 [11:39<02:30, 89.70it/s] 


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [38]:
def format_prompt_response(prompt: str, response: str) -> str:
    # Simple format. You may replace this with Qwen chat template later.
    return f"User: {prompt}\nAssistant: {response}"

@torch.no_grad()
def conditional_logprob(model, tokenizer, prompt: str, response: str, max_total: int = 512) -> float:
    prompt_text = f"User: {prompt}\nAssistant:"
    full_text = f"{prompt_text} {response}"

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=max_total, return_tensors="pt")

    input_ids = full["input_ids"].to(model.device)
    attention_mask = full["attention_mask"].to(model.device)

    # If prompt was truncated weirdly, skip.
    prompt_len = len(prompt_ids)
    if prompt_len >= input_ids.shape[1]:
        return float("-inf")

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    labels = input_ids[:, 1:]

    logprobs = torch.log_softmax(logits, dim=-1)
    token_logprobs = logprobs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    # Response tokens start after prompt_len.
    # Because labels are shifted, token at original position j corresponds to labels position j-1.
    start = max(prompt_len - 1, 0)
    resp_logprobs = token_logprobs[:, start:]

    return float(resp_logprobs.sum().item())

def base_preference_accuracy(rows: List[Dict[str, Any]], n: int = 200):
    sample = rows[:]
    random.Random(SEED).shuffle(sample)
    sample = sample[:min(n, len(sample))]

    correct = 0
    margins = []

    for r in tqdm(sample, desc="base pref acc"):
        lp_c = conditional_logprob(base_model, tok, r["prompt"], r["chosen"], max_total=MAX_TOTAL)
        lp_r = conditional_logprob(base_model, tok, r["prompt"], r["rejected"], max_total=MAX_TOTAL)

        margin = lp_c - lp_r
        margins.append(margin)
        correct += int(margin > 0)

    return {
        "n": len(sample),
        "pref_acc": correct / max(len(sample), 1),
        "mean_margin": float(np.mean(margins)),
        "median_margin": float(np.median(margins)),
    }



In [39]:
base_acc_report = base_preference_accuracy(rows_filtered, n=200)
base_acc_report


base pref acc:   0%|          | 0/200 [00:00<?, ?it/s]We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


base pref acc: 100%|██████████| 200/200 [00:21<00:00,  9.26it/s]


{'n': 200,
 'pref_acc': 0.545,
 'mean_margin': 5.975081577301025,
 'median_margin': 5.241039276123047}